# 02 · 네 가지 전략
**2세션 · 네 가지 전략** — 실습 10분

창이 커져도 **보낸 토큰만큼 비용·시간이 들고, 관련 없는 내용이 많으면 정확도가 떨어집니다** — 그래서 창 크기와 무관하게 넣는 방법을 고민해야 합니다. 방법은 네 가지뿐입니다:

- **고르기** — 전체가 아니라 필요한 부분만 검색해서 넣는다 (01의 B, = RAG)
- **줄이기** — 대화가 길어지면 오래된 부분을 요약으로 줄여서 넣는다
- **쓰기** — 알아낸 것을 창 밖(파일)에 적어 두고, 필요할 때만 다시 읽어서 넣는다
- **나누기** — 일을 서브에이전트에게 나눠, 각자 자기 창에서 처리하고 요약만 돌려받는다

오늘 깊게 하는 것은 *고르기*(3세션)이고, 여기서는 **줄이기·쓰기**를 작게 맛봅니다. *나누기*는 내일(에이전트) 주제라 오늘은 이름만 둡니다.

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")

## 1. 줄이기 — 대화가 길어지면
모델은 기억하지 않으므로, 이어 가려면 이력을 매번 다시 보냅니다. 입력 토큰이 어떻게 늘어나는지 봅니다.

In [ ]:
history = []
questions = ["연차 신청 절차를 알려줘", "반차는 어떻게 나뉘어?", "경조휴가 기준은?", "병가는 며칠까지야?", "방금 말한 것들을 표로 정리해줘"]
for q in questions:
    r = llm.call(q, history=history, max_tokens=300)
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": r.text}]
    print(f"{q:<22} 입력 {r.input_tokens:>5} 토큰")

**줄이기**: 직전 몇 턴은 그대로 두고, 그 이전은 요약 한 덩어리로 바꿉니다.

In [ ]:
old, recent = history[:-4], history[-4:]
old_text = "\n".join(f"{m['role']}: {m['content']}" for m in old)
summary = llm.call("다음 대화에서 나중에 필요할 사실만 다섯 줄 이내로 요약해 주세요.\n\n" + old_text, max_tokens=300).text

short = [{"role": "user", "content": "<이전 대화 요약>\n" + summary + "\n</이전 대화 요약>"},
         {"role": "assistant", "content": "네, 요약을 참고하겠습니다."}] + recent
q = "지금까지 이야기한 휴가 종류를 한 줄씩 다시 말해줘"
a = llm.call(q, history=history, max_tokens=300)
b = llm.call(q, history=short, max_tokens=300)
print(f"전체 이력  입력 {a.input_tokens} 토큰\n요약 + 최근  입력 {b.input_tokens} 토큰\n")
print(b.text)

## 2. 쓰기 — 창 밖에 적어 두고 필요할 때 읽는다
작업 중 알게 된 것을 파일에 써 두면, 다음 호출(또는 다음 날)에 그 파일만 창에 넣으면 됩니다.
내일 Claude Code의 `CLAUDE.md`가 바로 이 전략입니다.

In [ ]:
notes = Path("results/notes.md")
notes.write_text("# 작업 메모\n- 사용자는 생산관리팀 소속\n- 답은 표로 받기를 원함\n- 연차 규정은 일반 업무 규칙 제13~15조\n", encoding="utf-8")

r = llm.call("반차 규정을 알려줘", system="<메모>\n" + notes.read_text(encoding="utf-8") + "</메모>", max_tokens=300)
print(r.text)

## 3. 실습 · 내 업무에 전략 붙이기 (10분)
각자 업무 두 가지를 적고 전략을 고릅니다. 두세 명이 발표합니다.

| 내 업무 | 모델이 몰라서 틀릴 것 | 주로 쓸 전략 | 창에 넣을 것 |
|---|---|---|---|
| | | | |
| | | | |

> 대부분의 업무는 **고르기**에서 시작한다는 것을 확인하고 3세션으로 넘어갑니다.